# 07: ARBITER revision campaign (tolerance 1e-12 and replacement experiments)

**Run this notebook only.** Notebooks 00 to 06 stay as they are; this
notebook reads their audited results from Google Drive without changing
them, verifies every file by SHA-256, and then:

1. re-runs every experiment whose result depends on the verifier
   tolerance, at 1e-12 (with 1e-7 and 1e-9 as measured comparisons);
2. runs four replacement experiments: declaration coverage, seller
   pricing shapes, recompute versus verify, and buyer history;
3. re-checks the release protocol at 1e-12;
4. writes one evidence ZIP with a claim ledger and checksums.

**How to run:** first with `RUN_MODE = "pilot"` (Runtime > Run all).
When the last cell prints `PILOT PASS` and `GO`, change only
`RUN_MODE` to `"full"`, choose Runtime > Disconnect and delete runtime,
and Run all again. If Colab disconnects during the full run, reconnect
and Run all: finished parts are reloaded from Drive, not recomputed.


In [ ]:
# ======================= USER SETTINGS =======================
# Run once with RUN_MODE = "pilot". After the pilot prints
# "PILOT PASS" and "GO", change ONLY the next line to "full",
# choose Runtime > Disconnect and delete runtime, then Run all.
RUN_MODE = "pilot"            # "pilot" or "full"
ENVIRONMENT_LABEL = "colab_cpu_primary"
MANUAL_V4_PACKAGE = None      # leave None unless asked
TAKE_OVER_LOCK = False        # leave False unless asked
# =============================================================

import base64
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import secrets
import subprocess
import sys
import tempfile
import time

NOTEBOOK_NAME = "07_Arbiter_Revision_Campaign_v1_0.ipynb"
LOCAL_TEST = os.environ.get("ARBITER_LOCAL_TEST") == "1"
if LOCAL_TEST:
    RUN_MODE = os.environ.get("ARBITER_RUN_MODE_OVERRIDE", RUN_MODE)
if RUN_MODE not in {"pilot", "full"}:
    raise ValueError('RUN_MODE must be "pilot" or "full".')
print(f"Notebook 07 setup: RUN_MODE = {RUN_MODE}")

# 1. OSQP, the only package this notebook installs. Everything else is
#    already in Colab; versions are recorded in environment.json.
def ensure_osqp():
    if LOCAL_TEST:
        sys.path.insert(0, os.environ["ARBITER_LOCAL_TEST_DOUBLE_DIR"])
        import osqp_test_double
        sys.modules["osqp"] = osqp_test_double
        return "test-double"
    try:
        current = importlib.metadata.version("osqp")
    except importlib.metadata.PackageNotFoundError:
        current = None
    if current != "1.0.4":
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "osqp==1.0.4"])
        except subprocess.CalledProcessError:
            print("WARNING: osqp==1.0.4 did not install; installing the newest OSQP 1.x instead.")
            print("The claim audit will record the OSQP version actually used.")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "osqp>=1.0,<2"])
    importlib.invalidate_caches()
    import osqp  # noqa: F401
    return importlib.metadata.version("osqp")

OSQP_VERSION = ensure_osqp()
print("OSQP:", OSQP_VERSION)

# 2. Google Drive.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    drive.mount("/content/drive")
    default_root = "/content/drive/MyDrive/Arbiter_ICDE_2027"
else:
    default_root = "/content/Arbiter_ICDE_2027"
PROJECT_ROOT = Path(os.environ.get("ARBITER_PROJECT_ROOT", default_root))
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"{PROJECT_ROOT} does not exist. Notebook 07 needs the Drive folder "
        "MyDrive/Arbiter_ICDE_2027 created by Notebooks 00 to 06."
    )
print("Project folder:", PROJECT_ROOT)

# 3. Embedded source files, verified byte for byte before import.
embedded_modules = {"arbiter_closure_experiments.py": "...", "arbiter_core.py": "...", "arbiter_experiments.py": "...", "arbiter_request_release.py": "...", "arbiter_revision.py": "...", "arbiter_workload.py": "..."}
expected_hashes = {
    "arbiter_closure_experiments.py": "bf63760ee2124e6976bcf02e17d14c00c083defa4bdedae70fedfdec46e7cfb4",
    "arbiter_core.py": "b6797381dd883eac5d3cef7f1f86eaa2ad66bf73c498fdcfb7493f5113175ffa",
    "arbiter_experiments.py": "f5919e83b4c2658950a611e94f3bc8de62000a4ab7bd1cb24b90f64e5a0b590e",
    "arbiter_request_release.py": "b00dc16f4ab5ea9fc57d30fb141c83f31003bc8014e8285c1e1eabd812f66fc3",
    "arbiter_revision.py": "8ed288e64963bf44155f99529866adeab2931ca5ca97a6f027d6279e67e26e4d",
    "arbiter_workload.py": "cd3a78b28c573698e77fe02bc301273bbc959d770ee6daadae52ace68151ef5a"
}
# Keep one session identity when cells are rerun in the same runtime.
SESSION_ID = globals().get("SESSION_ID") or secrets.token_hex(8)
LOCAL_CODE_DIR = Path(tempfile.mkdtemp(prefix=f"arbiter-revision-code-{SESSION_ID}-"))
EMBEDDED_TEXT = {}
for name, encoded in embedded_modules.items():
    data = base64.b64decode(encoded)
    if hashlib.sha256(data).hexdigest() != expected_hashes[name]:
        raise RuntimeError(f"Embedded module hash mismatch: {name}. Re-download the notebook.")
    (LOCAL_CODE_DIR / name).write_bytes(data)
    EMBEDDED_TEXT[name] = data.decode("utf-8")
for module_name in [
    "arbiter_revision", "arbiter_closure_experiments", "arbiter_request_release",
    "arbiter_experiments", "arbiter_workload", "arbiter_core", "arbiter_data",
]:
    sys.modules.pop(module_name, None)
sys.path.insert(0, str(LOCAL_CODE_DIR))
importlib.invalidate_caches()
import arbiter_core
if arbiter_core.osqp is None:
    raise RuntimeError("OSQP did not load; the QP baseline cannot run. Restart the runtime and Run all.")
import arbiter_revision as rev
for module in [arbiter_core, rev]:
    if Path(module.__file__).resolve().parent != LOCAL_CODE_DIR.resolve():
        raise ImportError(f"{module.__name__} loaded from an unexpected place: {module.__file__}")
print("Startup import self-check PASS")

# 4. Read-only audited-v4 input.
EXTRACT_ROOT = Path(tempfile.mkdtemp(prefix=f"arbiter-v4-input-{SESSION_ID}-"))
PACKAGE_DIR, PACKAGE_SOURCE = rev.locate_audited_package(PROJECT_ROOT, MANUAL_V4_PACKAGE, EXTRACT_ROOT)
INPUT_AUDIT = rev.verify_audited_package(PACKAGE_DIR)
PACKAGE_STATE_BEFORE = rev.snapshot_folder_state(PACKAGE_DIR)
print(f"Audited-v4 input integrity PASS ({int(INPUT_AUDIT['passed'].sum())} checks, source: {PACKAGE_SOURCE})")

# 5. Output folder, lock, and experiment context.
override = None
if LOCAL_TEST and os.environ.get("ARBITER_CONFIG_OVERRIDE"):
    override = json.loads(os.environ["ARBITER_CONFIG_OVERRIDE"])
OSQP_RULE = None
if RUN_MODE == "full":
    OSQP_RULE = rev.full_mode_osqp_rule(PROJECT_ROOT, ENVIRONMENT_LABEL)
    override = {**(override or {}), "osqp_qp_repetitions": OSQP_RULE["osqp_qp_repetitions"],
                "osqp_qp_warmups": OSQP_RULE["osqp_qp_warmups"]}
    print(
        f"Pilot OSQP cost at {OSQP_RULE['largest_pilot_scale']} assets: "
        f"{OSQP_RULE['pilot_osqp_mean_ms_at_largest_scale']:.0f} ms per call; the full run uses "
        f"{OSQP_RULE['osqp_qp_repetitions']} OSQP timing repetitions in stage s04 (rule fixed in advance)."
    )
    print(f"The pilot estimated about {OSQP_RULE['pilot_estimated_full_hours']:.1f} hours for this full run.")
CONFIG = rev.mode_config(RUN_MODE, override)
if OSQP_RULE is not None:
    CONFIG["osqp_repetition_rule"] = OSQP_RULE
OUTPUT_DIR = PROJECT_ROOT / "runs" / "revision_v1" / f"07_revision_{RUN_MODE}_{rev.slug(ENVIRONMENT_LABEL)}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCK = rev.RunLock(OUTPUT_DIR / "run.lock", session_id=SESSION_ID, stale_seconds=300, take_over=TAKE_OVER_LOCK)
LOCK.acquire()
try:
    ctx = rev.build_context(
        mode=RUN_MODE, config=CONFIG, package_dir=PACKAGE_DIR, output_dir=OUTPUT_DIR,
        input_audit=INPUT_AUDIT, package_state_before=PACKAGE_STATE_BEFORE,
        session_id=SESSION_ID, lock=LOCK,
    )
except Exception:
    LOCK.release()
    raise
finger = rev.runtime_fingerprint(SESSION_ID)
print(f"CPU: {finger['cpu_model']} ({finger['logical_cpu_count']} logical CPUs); Python {finger['python']}")
print("Output folder:", OUTPUT_DIR)
print("Setup PASS. Stages already finished in an earlier session are reloaded, not rerun.")


## Stage 01: Natural proposals: tolerance 1e-7 / 1e-9 / 1e-12, leftover savings, heavy-load baselines


In [ ]:
# Stage s01: Natural proposals: tolerance 1e-7 / 1e-9 / 1e-12, leftover savings, heavy-load baselines
display(rev.run_stage_guarded(ctx, "s01"))


## Stage 02: Exact per-row box attacks at 1e-12


In [ ]:
# Stage s02: Exact per-row box attacks at 1e-12
display(rev.run_stage_guarded(ctx, "s02"))


## Stage 03: Omitted-relation gap at 1e-12


In [ ]:
# Stage s03: Omitted-relation gap at 1e-12
display(rev.run_stage_guarded(ctx, "s03"))


## Stage 04: Scale benchmark (128 to 4,096 assets) with paired baselines at 1e-12


In [ ]:
# Stage s04: Scale benchmark (128 to 4,096 assets) with paired baselines at 1e-12
display(rev.run_stage_guarded(ctx, "s04"))


## Stage 05: Relation-family and active-row-budget ablation at 1e-12


In [ ]:
# Stage s05: Relation-family and active-row-budget ablation at 1e-12
display(rev.run_stage_guarded(ctx, "s05"))


## Stage 06: Dense active-row budgets at 1e-7 and 1e-12


In [ ]:
# Stage s06: Dense active-row budgets at 1e-7 and 1e-12
display(rev.run_stage_guarded(ctx, "s06"))


## Stage 07: Fallback kernels at 1e-12


In [ ]:
# Stage s07: Fallback kernels at 1e-12
display(rev.run_stage_guarded(ctx, "s07"))


## Stage 08: Declaration coverage (replaces the declaration-set table)


In [ ]:
# Stage s08: Declaration coverage (replaces the declaration-set table)
display(rev.run_stage_guarded(ctx, "s08"))


## Stage 09: Seller pricing shapes: linear, concave, tiered (replaces the pricing-shape table)


In [ ]:
# Stage s09: Seller pricing shapes: linear, concave, tiered (replaces the pricing-shape table)
display(rev.run_stage_guarded(ctx, "s09"))


## Stage 10: Recompute versus verify (replaces the recompute table)


In [ ]:
# Stage s10: Recompute versus verify (replaces the recompute table)
display(rev.run_stage_guarded(ctx, "s10"))


## Stage 11: Buyer history (replaces the buyer-history row)


In [ ]:
# Stage s11: Buyer history (replaces the buyer-history row)
display(rev.run_stage_guarded(ctx, "s11"))


## Stage 12: Protocol vectors and fault campaigns at 1e-12


In [ ]:
# Stage s12: Protocol vectors and fault campaigns at 1e-12
display(rev.run_stage_guarded(ctx, "s12"))


## Final audit and evidence ZIP


In [ ]:
# Final audit, claim ledger, and evidence ZIP.
result = rev.finalize_campaign(
    ctx,
    embedded_sources=EMBEDDED_TEXT,
    embedded_hashes=expected_hashes,
    notebook_name=NOTEBOOK_NAME,
    environment_label=ENVIRONMENT_LABEL,
)
quality = result["quality"]
display(quality[~quality["passed"].astype(bool)] if (~quality["passed"].astype(bool)).any() else quality.tail(5))
display(result["claims"][["claim", "status", "value"]])
if RUN_MODE == "pilot":
    estimate = rev.read_csv(OUTPUT_DIR / "runtime_estimate.csv")
    display(estimate[["stage", "description", "pilot_seconds", "estimated_full_minutes"]])
    total_hours = float(estimate["estimated_full_seconds"].sum()) / 3600.0
    limit = float(CONFIG["runtime_go_limit_hours"])
    print(f"Estimated full run: {total_hours:.1f} hours (planning estimate, not a measurement).")
LOCK.release()
print("Evidence ZIP:", result["zip_path"])
print("Size:", result["zip_bytes"], "bytes")
if result["status"] != "PASS":
    raise RuntimeError(
        "Notebook 07 finished with failed checks: " + "; ".join(result["fatal_failures"])
        + ". Do not start the full run. Send this ZIP and the error text."
    )
if RUN_MODE == "pilot":
    if total_hours <= limit:
        print(f"PILOT PASS. GO: change RUN_MODE to \"full\", disconnect and delete the runtime, then Run all.")
    else:
        print(f"PILOT PASS, but NO-GO: the estimate exceeds {limit:.0f} hours. Send the pilot ZIP before running full.")
else:
    print("FULL RUN PASS. Download the ZIP above and send it for the audit.")
